# Hướng dẫn Chạy mô hình Multimodal (Late Fusion) trên Google Colab
Notebook này đã được cấu hình tự động 100% để bạn có thể chạy từ đầu đến cuối mà không gặp lỗi.

**LƯU Ý TRƯỚC KHI CHẠY:**
Bạn cần đảm bảo đã tạo thư mục Lối tắt (Shortcut) tên là `SE365` trên Google Drive của bạn (trỏ từ link chia sẻ dữ liệu của nhóm). Nếu bạn đặt tên lối tắt khác, hãy sửa tên thư mục ở **Bước 3** và **Bước 3.5** 

### BƯỚC 1: Mount Google Drive
Lệnh này sẽ yêu cầu bạn cấp quyền truy cập Google Drive. Chúng ta cần làm điều này để đọc dữ liệu gốc (5000 ảnh và CSV) thông qua Lối tắt (Shortcut) mà không cần tải lại file zip.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### BƯỚC 2: Clone mã nguồn và cài đặt thư viện
Tải phiên bản code mới nhất từ Github và cài đặt các thư viện cần thiết (PyTorch, Transformers, Timm, ...).


In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt

Cloning into 'SE365'...
remote: Enumerating objects: 12900, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 12900 (delta 57), reused 97 (delta 31), pack-reused 12774 (from 1)
Receiving objects: 100% (12900/12900), 867.85 MiB | 23.47 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/SE365


### BƯỚC 3: Nạp dữ liệu vào máy ảo Colab (QUAN TRỌNG)
Để GPU không phải chờ ổ cứng đọc qua mạng, chúng ta sẽ copy toàn bộ thư mục `data` từ Drive sang thẳng ổ cứng cục bộ của máy ảo Colab. Việc này sẽ mất khoảng 1-2 phút nhưng tăng tốc huấn luyện lên gấp nhiều lần.

**Chỗ cần sửa:** Nếu lối tắt của bạn không lưu ở thư mục gốc Drive hoặc có tên khác `SE365`, hãy sửa đường dẫn `/content/drive/MyDrive/SE365/data` bên dưới cho khớp.

In [ ]:
!rm -rf ./data
# SỬA LẠI ĐƯỜNG DẪN NẾU CẦN (Đảm bảo đường dẫn đúng với thư mục chứa data trên Google Drive của bạn)
!cp -r /content/drive/MyDrive/SE365/data ./data

# Kiểm tra xem dữ liệu đã được copy chưa
!ls -la ./data

total 304
drwx------  4 root root   4096 Jun 11 08:34 .
drwxr-xr-x 11 root root   4096 Jun 11 08:34 ..
drwx------  2 root root 299008 Jun 11 08:39 image
drwx------  2 root root   4096 Jun 11 08:34 text


### BƯỚC 3.5: Cấu hình nơi lưu Checkpoint (Trọng số mô hình)
Để tránh bị mất kết quả khi Colab tự ngắt, đoạn code sau sẽ tạo một thư mục con trong `checkpoints/` trên Drive của bạn, với tên là thời gian hiện tại (Ví dụ: `20260611_083908`). 
Tất cả các mô hình Text, Image, Fusion sau khi train xong sẽ được tự động copy vào chung thư mục này.

**Chỗ cần sửa:** Nếu bạn muốn lưu vào thư mục khác, hãy sửa biến `drive_ckpt_path`.


In [5]:
# BƯỚC 3.5: Khởi tạo thư mục lưu trữ cho toàn bộ phiên chạy này
# Đảm bảo tất cả mô hình train trong hôm nay đều nằm chung một thư mục
import os
import datetime

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
drive_ckpt_path = f'/content/drive/MyDrive/SE365/checkpoints/{run_id}'
os.environ['DRIVE_CKPT'] = drive_ckpt_path

# Tạo thư mục con checkpoints và thư mục run_id bằng os.makedirs
os.makedirs(drive_ckpt_path, exist_ok=True)
print(f'Mọi checkpoint trong phiên này sẽ được lưu chung vào: {drive_ckpt_path}')

Mọi checkpoint trong phiên này sẽ được lưu chung vào: /content/drive/MyDrive/SE365/checkpoints/20260611_083908


### BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
**Các tham số bạn có thể tuỳ chỉnh ở lệnh Train (cả 3 mô hình text, imame, fusion):**
- `--epochs`: Số vòng lặp (Mặc định: 5)
- `--batch_size`: Kích thước batch (Mặc định: 16)
- `--lr`: Learning rate (Mặc định: 2e-5)
- `--alpha`: Trọng số loss cho các yếu tố phụ (Mặc định: 0.5)
- Đường dẫn data: `--train_path`, `--val_path`, `--test_path`, `--image_dir`


In [6]:
# BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
!python main.py --mode train_text --epochs 1

# Lưu Checkpoint Text ngay lập tức
!cp ./checkpoints/best_model_train_text.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Text vào $DRIVE_CKPT"

====== MODE: TRAIN_TEXT ======
Using device: cuda
config.json: 100% 615/615 [00:00<00:00, 2.49MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 126kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 12.4MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 22.1MB/s]
preprocessor_config.json: 100% 266/266 [00:00<00:00, 1.53MB/s]
Loading Dataset...
Đã nạp 3971 mẫu cho Train và 501 mẫu cho Val
Initializing Model...
model.safetensors: 100% 1.12G/1.12G [00:21<00:00, 51.8MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 8170.35it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

### BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)

In [7]:
# BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)
!python main.py --mode train_image --epochs 1

# Lưu Checkpoint Image ngay lập tức
!cp ./checkpoints/best_model_train_image.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Image vào $DRIVE_CKPT"

====== MODE: TRAIN_IMAGE ======
Using device: cuda
Loading Dataset...
Đã nạp 3971 mẫu cho Train và 501 mẫu cho Val
Initializing Model...
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
model.safetensors: 100% 440M/440M [00:08<00:00, 50.1MB/s]
Epoch 1/1: 100% 249/249 [05:20<00:00,  1.29s/it, loss=1.76]

Epoch 1/1 | Train Loss: 13.2440 | Val Loss: 2.7663
  -> RMSE | Overall: 1.5270 | Food: 1.8194 | Price: 1.7851 | Atmos: 1.7622
  -> MAE  | Overall: 1.0627 | Food: 1.2879 | Price: 1.2690 | Atmos: 1.3056
*** Saved best model to ./checkpoints/best_model_train_image.pth ***

Đã lưu checkpoint Image vào /content/drive/MyDrive/SE365/checkpoints/20260611_083908


In [8]:
# BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
!python main.py --mode train_fusion --epochs 1

# Lưu Checkpoint Fusion ngay lập tức
!cp ./checkpoints/best_model_train_fusion.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Fusion vào $DRIVE_CKPT"

====== MODE: TRAIN_FUSION ======
Using device: cuda
Loading Dataset...
Đã nạp 3971 mẫu cho Train và 501 mẫu cho Val
Initializing Model...
Loading weights: 100% 199/199 [00:00<00:00, 746.34it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(
Loaded Text Model weights from ./checkpoints/best_model_train_text.pth
Loaded Image Model weights from ./checkpoints

In [9]:
# BƯỚC 7: Đánh giá mô hình (Tính các metric MAE, MSE, RMSE)
!python test.py --mode train_fusion

====== TESTING MODE: TRAIN_FUSION ======
Loading Test Dataset...
Đã nạp 528 mẫu Test
Loading weights: 100% 199/199 [00:00<00:00, 887.13it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = create_fn(

[EVALUATION METRICS ON INDEPENDENT TEST SET]
MSE  (Overall): 1.8656 | (Food): 2.5731 | (Price): 2.6575 | (Atmos): 2.9387
RMSE (Overall): 1.3659 | (Food): 1.6041 | (Pric